# STAC quickstart — Sentinel-2 COGs from Earth Search

The `earthlens.stac` backend searches a STAC API, mosaics the matching items to your bbox, reprojects/stacks the requested bands, and writes one Cloud-Optimized GeoTIFF per `(collection, date)`. Earth Search (Element 84 / AWS) is anonymous, so this needs only `pip install earthlens[stac]` — no account.

The download cell runs a **real** request against Earth Search; if the network is down or the upstream is broken, the cell will error so you can see why.

## The catalog (no network)

A collection is addressed by a logical key; the catalog resolves it to the id each endpoint actually serves.

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
print("endpoints:", sorted(cat.endpoints))
print("earth-search id:", cat.resolve("earth-search", "sentinel-2-l2a"))
print("default assets:", cat.get_collection("sentinel-2-l2a").default_assets)

## Download a Sentinel-2 RGB scene over Madrid (live)

Earth Search serves Sentinel-2 L2A assets under common names (`red`, `green`, `blue`). `max_items=1` keeps the quickstart to a single scene.

In [ ]:
import tempfile

from earthlens.core import EarthLens

el = EarthLens(
    data_source="earth-search",
    start="2024-06-01",
    end="2024-06-20",
    dataset="sentinel-2-l2a",
    variables=["red", "green", "blue"],
    aoi=[-3.72, 40.40, -3.67, 40.45],
    path=tempfile.mkdtemp(),
    max_items=1,
)
paths = el.download()
paths

## Inspect the written COG

Open it with pyramids and show its shape / CRS / bands.

In [ ]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(paths[0])
arr = ds.read_array()
print("bands:", ds.band_count, "epsg:", ds.epsg, "shape:", arr.shape)

Swap `data_source` for `"planetary-computer"` (asset keys `B04`/`B03`/`B02`) or `"cdse"` (needs S3 keys — see Authentication) to pull the same AOI from another provider. Use `aggregate=AggregationConfig(freq="1MS")` for per-month composites.